# Pooled current-sheet thickness in Luca's simulations

The same pooled thickness as the beta-scan notebooks, for the five runs on the /DATA disk. They differ from our
runs in their file tag, their grid spacing (0.125 $d_p$), their snapshots, and their filter: the older scheme,
`N_IMFS` IMFs of `iterative_filtering3D`, rather than a cutoff wavenumber.

This replaces the 440-line copy of the pipeline the notebook used to hold.

In [ ]:
import csv

import matplotlib.pyplot as plt

import pta
from pta import config
from pta.plotting import use_style
from pta.sheets import pooled_thickness

use_style()
RESULTS = config.results_dir("cs_identification", "LucaSimulationAnalysis")

In [ ]:
# One entry per simulation; `path` is relative to the external_data folder in pta.toml (/DATA).
SIMULATIONS = [
    dict(run=1, path="QMUL/data-franci/data/RunCompa_2Dbig_rms025_modes4", tag="EL4modes",
         snapshots=[48, 56, 64, 72, 80]),
    dict(run=2, path="Nancy_S3_moreppc", tag="EL16modesMorePPC", snapshots=[48, 56, 64, 72, 80]),
    dict(run=3, path="Nancy_S3_moreppc_lowEta", tag="EL16mLowRes", snapshots=[48, 56, 64, 72, 80]),
    dict(run=4, path="QMUL/data-franci/data/RunEL_beta0.5_n2048_dx0.125_ppc1024_rms0.25_modes16",
         tag="EL16modes", snapshots=[12, 14, 16, 18, 20]),
    dict(run=5, path="QMUL/data-franci/data/RunCompa_2Dbig_rms012_modes8", tag="EL8modes",
         snapshots=[48, 56, 64, 72, 80]),
]
SPACING = 0.125                       # grid spacing of all five runs, in d_p
N_IMFS, XI = 1, 3.0                   # the older iterative filter, as used for these runs
RUN_ONLY = None                       # e.g. [2, 4] to analyse only some of them

METHODS = ("m1", "m2", "m3")
N_BOOTSTRAP = 1000

In [ ]:
summary = []
for simulation in SIMULATIONS:
    if RUN_ONLY and simulation["run"] not in RUN_ONLY:
        continue
    run = pta.Run(config.path("external_data") / simulation["path"], tag=simulation["tag"], spacing=SPACING)
    pooled = pooled_thickness(run=run, times=simulation["snapshots"], n_iter=N_IMFS, Xi=XI,
                              method=METHODS, bootstrap=N_BOOTSTRAP,
                              title=rf"Pooled current-sheet thickness   run {simulation['run']} ({simulation['tag']})")
    folder = config.results_dir("cs_identification", "LucaSimulationAnalysis", str(simulation["run"]))
    figure_path, array_path = pooled.save(folder, f"pooled_thickness_run{simulation['run']}")
    print(f"run {simulation['run']} ({simulation['tag']}): {pooled.sheets_per_time}\n  -> {figure_path}")
    plt.show()
    summary += [dict(run=simulation["run"], **row) for row in pooled.statistics]

In [ ]:
# One row per run and method
columns = ["run", "method", "n_snapshots", "n", "median_lognorm", "sigma_lognorm", "median_se", "ci_lo", "ci_hi",
           "mean", "median_emp"]
with open(RESULTS / "pooled_summary.csv", "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(summary)
print(f"saved {RESULTS / 'pooled_summary.csv'}")